# NexaTel CDR Pipeline — Task 4: Error Handler

> **Industry context — NexaTel Communications**  
> When the CDR pipeline fails, the billing and customer experience teams need  
> an immediate record of what went wrong. This notebook is **Task 4** in the  
> Lakeflow Job. It only runs when an upstream task fails, logs the failure  
> summary, and triggers additional alerting.

**Lakeflow Job DAG position:** `cleanse_cdr` → **`error_handler`**  
**Depends on:** `cleanse_cdr` (run-if condition: **At least one failed**)  

**Trainer note:** In the Lakeflow Jobs UI, set this task's  
"Depends on" → `cleanse_cdr` and set **Run if** to **At least one failed**.  
This task will be **skipped** when `cleanse_cdr` succeeds.

In [ ]:
# ── CELL 1: Collect context about what failed ─────────────────────────────────
# Trainer: dbutils.jobs.taskValues lets us read what upstream tasks logged.
# Even if cleanse_cdr failed, ingest_cdr may have stored useful context.

dbutils.widgets.text("billing_month",  "2024-12", "Billing Month")
dbutils.widgets.text("source_catalog", "trainer_demo", "Source Catalog")
dbutils.widgets.text("source_schema",  "demo_11",       "Source Schema")

billing_month  = dbutils.widgets.get("billing_month")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")

context = {"billing_month": billing_month}

try:
    context["total_raw_records"] = dbutils.jobs.taskValues.get(
        taskKey="ingest_cdr", key="total_raw_records"
    )
except Exception:
    context["total_raw_records"] = "(not available — ingest_cdr may also have failed)"

print("=== Error Handler: pipeline failure context ===")
for k, v in context.items():
    print(f"  {k}: {v}")

In [ ]:
# ── CELL 2: Inspect raw data for obvious issues ───────────────────────────────
# Trainer: when a pipeline fails we often want a quick diagnostic snapshot.
# This cell shows what data looked like before cleansing.

from pyspark.sql import functions as F
from datetime import datetime

try:
    year, month = billing_month.split("-")
    raw_cdr = (
        spark.read.table(f"{source_catalog}.{source_schema}.raw_cdr")
        .filter(
            (F.year("call_start_ts")  == int(year)) &
            (F.month("call_start_ts") == int(month))
        )
    )

    null_subscribers = raw_cdr.filter(F.col("subscriber_id").isNull()).count()
    bad_durations    = raw_cdr.filter(F.col("duration_seconds") <= 0).count()
    total_raw        = raw_cdr.count()

    print(f"\nDiagnostic snapshot for {billing_month}:")
    print(f"  Total raw records      : {total_raw:,}")
    print(f"  NULL subscriber_id     : {null_subscribers:,}")
    print(f"  Invalid duration (<= 0): {bad_durations:,}")

    # Show a sample of problematic records
    print("\nSample of records with NULL subscriber_id:")
    raw_cdr.filter(F.col("subscriber_id").isNull()).show(3, truncate=False)

except Exception as e:
    print(f"Could not read raw_cdr for diagnostics: {e}")

In [ ]:
# ── CELL 3: Log failure to an audit / error-log table ─────────────────────────
# Trainer: production pipelines typically write failures to a dedicated log table
# so operations teams can query the history without digging through job run logs.

from pyspark.sql.types import StructType, StructField, StringType, TimestampType

log_schema = StructType([
    StructField("logged_at",        TimestampType(), True),
    StructField("billing_month",    StringType(),    True),
    StructField("failed_task",      StringType(),    True),
    StructField("error_summary",    StringType(),    True),
])

log_row = [(
    datetime.now(),
    billing_month,
    "cleanse_cdr",
    f"Pipeline failure detected. Raw records available: {context.get('total_raw_records', 'unknown')}"
)]

log_df = spark.createDataFrame(log_row, schema=log_schema)
log_df.write.format("delta").mode("append").saveAsTable(
    f"{source_catalog}.{source_schema}.pipeline_error_log"
)

print("✓ Error entry written to pipeline_error_log")
spark.read.table(f"{source_catalog}.{source_schema}.pipeline_error_log").show(truncate=False)

In [ ]:
# ── CELL 4: Exit with failure signal ─────────────────────────────────────────
# Trainer: dbutils.notebook.exit() propagates the status back to the Lakeflow Job.
# Using a non-SUCCESS string causes the job to mark this task as failed,
# which in turn triggers any downstream notifications (alerts unit demo).

print("\n⚠️  Task 4 (error_handler) complete — pipeline failure has been logged.")
print("   Check pipeline_error_log for the full error history.")
dbutils.notebook.exit("PIPELINE_FAILURE_LOGGED")